# Explore ERP datasets

In [ ]:
using Random

function find_repo_root(start_dir::AbstractString = pwd())
    candidates = unique(normpath.([
        start_dir,
        joinpath(start_dir, ".."),
        joinpath(start_dir, "..", ".."),
        joinpath(start_dir, "..", "..", ".."),
    ]))
    for candidate in candidates
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "scripts"))
            return candidate
        end
    end
    error("Could not locate repository root from start_dir=$(start_dir).")
end

const REPO_ROOT = find_repo_root()
const SCRIPTS_ENV_DIR = joinpath(REPO_ROOT, "scripts")
const BA2_DEPOT_DIR = joinpath(homedir(), ".julia_depots", "ba2")
mkpath(BA2_DEPOT_DIR)
if isempty(DEPOT_PATH) || first(DEPOT_PATH) != BA2_DEPOT_DIR
    filter!(path -> path != BA2_DEPOT_DIR, DEPOT_PATH)
    pushfirst!(DEPOT_PATH, BA2_DEPOT_DIR)
end
ENV["JULIA_NUM_PRECOMPILE_TASKS"] = "1"
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"
if !(SCRIPTS_ENV_DIR in LOAD_PATH)
    pushfirst!(LOAD_PATH, SCRIPTS_ENV_DIR)
end

include(joinpath(REPO_ROOT, "scripts", "erp_io.jl"))
include(joinpath(REPO_ROOT, "scripts", "erp_image_processing.jl"))


In [ ]:
function citation_summary(citation)
    if haskey(citation, "primary")
        primary = citation["primary"]
        return string(get(primary, "authors", "unknown"), " (", get(primary, "year", "n.d."), ")")
    elseif haskey(citation, "data")
        data = citation["data"]
        return string(get(data, "authors", "unknown"), " (", get(data, "year", "n.d."), ")")
    elseif haskey(citation, "note")
        return String(citation["note"])
    end
    return "citation metadata available"
end

println("Available datasets:")
for dataset_key in list_datasets()
    metadata = load_dataset_metadata(dataset_key)
    println(dataset_key, " | ", metadata["dataset_label"], " | ", citation_summary(metadata["citation"]))
end

In [ ]:
dataset_key = "fixations_dataset"

println("Sort variables available for $(dataset_key):")
for sort_variable in list_sort_variables(dataset_key)
    println(sort_variable)
end

In [ ]:
if !isdefined(Main, :ERPDataPlot)
    include(joinpath(REPO_ROOT, "scripts", "erp_plot.jl"))
end

function show_erp(dataset_key::AbstractString,
                  sort_variable::AbstractString,
                  channel_name::Union{Nothing,AbstractString} = nothing;
                  rng = Random.GLOBAL_RNG)
    channels = list_labeled_channels(dataset_key, sort_variable)
    isempty(channels) && error("No labeled channels found for dataset=$(dataset_key), sort_variable=$(sort_variable).")

    chosen = channel_name === nothing ? rand(rng, channels) : String(channel_name)
    if !(chosen in channels)
        error("Combination not labeled: channel=$(chosen), sort_variable=$(sort_variable). Available channels: $(join(channels, ", ")).")
    end

    return Base.invokelatest(plot_erp_image, dataset_key, chosen, sort_variable)
end

display(Base.invokelatest(show_erp, "fixations_dataset", "rt_ms"))
display(Base.invokelatest(show_erp, "fixations_dataset", "rt_ms", "ch042"))
